# Edit a SLURM header without changing the job body

**Author:** Haoyuan Li

This example parses a Stampede3 batch file, changes selected `#SBATCH` values and its launcher command, and writes a new file. Unmodified content is preserved exactly.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from gdmate.slurm import SlurmScript

Locate and parse the example relative to this notebook's directory. This works when the notebook is launched from either `notebooks` or the repository root.

In [ ]:
working_directory = Path.cwd()
notebook_directory = (
    working_directory
    if working_directory.name == "notebooks"
    else working_directory / "notebooks"
)
repository_root = notebook_directory.parent
example = repository_root / "scripts/server/stampede3/job_stampede3_2d.slurm"
script = SlurmScript.from_file(example)
[(directive.option, directive.value) for directive in script.directives]

Change existing header directives and inspect the detected `ibrun` command. The same command API also recognizes `srun` and `mpirun`. Here, the input parameter file is changed from `case.prm` to `case_modified.prm`.

In [ ]:
original_body = script.body
script.set_directive("--partition", "spr")
script.set_directive("-N", "2")
script.set_directive("--job-name", "my_2d_job")
original_command = script.command
print(f"Original command: {original_command}")
script.set_command(original_command.replace("case.prm", "case_modified.prm"))
print(f"Modified command: {script.command}")

Write a separate file and verify that reparsing it yields the modified command while every other body line remains unchanged. A temporary directory keeps the demonstration from leaving generated files in the repository.

In [ ]:
with TemporaryDirectory() as directory:
    output = Path(directory) / "job_stampede3_2d_modified.slurm"
    script.write(output)
    reparsed = SlurmScript.from_file(output)
    assert reparsed.command.endswith("case_modified.prm")
    assert reparsed.body == original_body.replace(original_command, script.command)
    print(reparsed.text)